# FactLedger extractor

`load(path) -> documents, units` over raw files and nothing else. The extractor sniffs the
format from the bytes and writes document and unit JSON in the shapes of SCHEMA.md; the
rules it follows are in BUILD.md. Built one block at a time. Inputs: the public raw dataset
and the private papers dataset, both attached to this notebook.


In [ ]:
# Block 1: inputs and integrity.
# Mount both datasets, count files per folder, and check every file's sha256 against the
# folder manifest. The manifests are used here only to prove the Kaggle copies are the bytes
# that were uploaded; the extractor itself never reads them.
import hashlib
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

def mount(slug):
    """Kaggle mounts inputs at /kaggle/input/<slug> or, in newer sessions,
    /kaggle/input/datasets/<owner>/<slug>. Take whichever exists."""
    for candidate in (Path("/kaggle/input") / slug, Path("/kaggle/input/datasets/jhffmn") / slug):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(slug)


RAW = mount("it494-narrative-corpora-raw")
PAPERS = mount("it494-reference-papers")


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def check(folder, rows_key):
    manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
    rows = manifest[rows_key]
    on_disk = {p.name for p in folder.iterdir() if p.name not in ("manifest.json", "LICENSE")}
    listed = {r["file"] for r in rows}
    # Kaggle inputs are a network filesystem: one file at a time, 19,206 files take tens of
    # minutes; 32 concurrent reads take about a minute.
    with ThreadPoolExecutor(max_workers=32) as pool:
        digests = list(pool.map(sha256, [folder / r["file"] for r in rows]))
    bad = [r["file"] for r, d in zip(rows, digests) if d != r["sha256"]]
    print(f"{folder.name:<28} files {len(on_disk):>6}  listed {len(listed):>6}"
          f"  mismatched {len(bad)}  unlisted {len(on_disk - listed)}  missing {len(listed - on_disk)}")
    return bad


# The three literature manifests keep their original "works" key; the unpacked folders
# and the papers use "files".
for name, key in [("oz", "works"), ("holmes", "works"), ("greek", "works"),
                  ("graphrag-bench", "files"), ("longmemeval", "files")]:
    check(RAW / name, key)
check(PAPERS, "files")


In [ ]:
# Block 2: file type and raw text.
# By bytes only: which container is this, and what is its text. Nothing here guesses what
# the text is about; that is the model call in block 3. Every document comes out as one
# string, the sha256 of the original bytes, and, for chat containers, the piece offsets code
# can see without a model: one piece per turn, with its time when the file carries one.
import json
import re
from datetime import datetime

try:
    import pymupdf  # the one dependency for the PDF text layer
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf"], check=True)
    import pymupdf

WORK = Path("/kaggle/working")


def decode(data):
    try:
        return data.decode("utf-8"), "utf-8", []
    except UnicodeDecodeError:
        return data.decode("cp1252", errors="replace"), "cp1252", ["not-utf8"]


def parse_time(value):
    """ISO 8601, or the yyyy/mm/dd (Dow) hh:mm form; None when it is neither."""
    if not isinstance(value, str):
        return None
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00")).isoformat()
    except ValueError:
        pass
    m = re.match(r"\s*(\d{4})/(\d{2})/(\d{2}).*?(\d{2}):(\d{2})", value)
    if m:
        y, mo, d, h, mi = (int(x) for x in m.groups())
        return datetime(y, mo, d, h, mi).isoformat()
    return None


def find_turns(obj):
    """The first list of dicts that all carry role and content, anywhere in the JSON."""
    if isinstance(obj, list) and obj and all(isinstance(t, dict) and "role" in t and "content" in t for t in obj):
        return obj
    if isinstance(obj, dict):
        for value in obj.values():
            found = find_turns(value)
            if found:
                return found
    return None


def block_text(content):
    """Message content as one string: a string, or a list of typed blocks."""
    if isinstance(content, str):
        return content
    parts = []
    for block in content or []:
        if not isinstance(block, dict):
            parts.append(str(block))
        elif block.get("type") == "text":
            parts.append(block.get("text", ""))
        elif block.get("type") == "tool_use":
            parts.append(f"[tool_use {block.get('name', '')}] {json.dumps(block.get('input', {}))}")
        elif block.get("type") == "tool_result":
            parts.append(f"[tool_result] {block_text(block.get('content', ''))}")
    return "\n".join(parts)


def render(header, turns):
    """Header lines, a blank line, then one 'role: content' block per turn. Returns the
    text and the pieces as offsets into it; the body is the run of pieces."""
    text = "\n".join(header) + "\n\n" if header else ""
    pieces = []
    for role, content, at in turns:
        start = len(text)
        text += f"{role}: {content}\n\n"
        pieces.append({"start": start, "end": len(text), "at": at})
    return text, pieces


def from_chat_json(obj, turns):
    fields = obj if isinstance(obj, dict) else {}
    header, dates = [], []
    for key in ("session_id", "id", "title"):
        if key in fields:
            header.append(f"{key}: {fields[key]}")
    raw_dates = fields.get("dates") or ([fields["date"]] if "date" in fields else [])
    for d in raw_dates:
        header.append(f"date: {d}")
        dates.append(parse_time(d))
    text, pieces = render(header, [(t["role"], block_text(t["content"]), parse_time(t.get("timestamp"))) for t in turns])
    flags = ["ambiguous-date"] if len(dates) > 1 else []
    occurred = dates[0] if len(dates) == 1 else None
    return text, pieces, occurred, raw_dates, flags


def from_chat_jsonl(rows):
    turns = []
    for row in rows:
        if row.get("type") in ("user", "assistant") and isinstance(row.get("message"), dict):
            turns.append((row["type"], block_text(row["message"].get("content")), parse_time(row.get("timestamp"))))
    header = [f"session: {rows[0]['sessionId']}"] if rows and "sessionId" in rows[0] else []
    text, pieces = render(header, turns)
    times = [p["at"] for p in pieces if p["at"]]
    return text, pieces, (min(times) if times else None), times[:1], []


def to_text(path):
    data = path.read_bytes()
    record = {"path": str(path), "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(),
              "kind": None, "encoding": None, "text": "", "pieces": None,
              "occurred_at": None, "dates": [], "flags": []}
    if data.startswith(b"%PDF-"):
        doc = pymupdf.open(stream=data, filetype="pdf")
        record.update(kind="pdf", encoding="pdf-text-layer", text="\n".join(page.get_text() for page in doc))
        if not record["text"].strip():
            record["flags"].append("no-text-layer")
        return record
    text, encoding, flags = decode(data)
    record.update(encoding=encoding, flags=flags)
    obj = None
    try:
        obj = json.loads(text)
    except ValueError:
        pass
    if obj is not None:
        turns = find_turns(obj)
        if turns:
            record["text"], record["pieces"], record["occurred_at"], record["dates"], more = from_chat_json(obj, turns)
            record["kind"] = "chat-json"
            record["flags"] += more
            return record
        record.update(kind="text", text=text)
        record["flags"].append("json-not-chat")
        return record
    rows = []
    for line in text.splitlines():
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except ValueError:
            rows = None
            break
    if rows and all(isinstance(r, dict) for r in rows) and any(r.get("type") in ("user", "assistant") for r in rows):
        record["text"], record["pieces"], record["occurred_at"], record["dates"], more = from_chat_jsonl(rows)
        record["kind"] = "chat-jsonl"
        record["flags"] += more
        return record
    record.update(kind="text", text=text)
    return record


def describe(record):
    head = record["text"][:60].replace("\n", " / ")
    pieces = "-" if record["pieces"] is None else len(record["pieces"])
    return (f"{Path(record['path']).name:<34} {record['kind']:<10} {record['bytes']:>9,} b "
            f"{len(record['text']):>9,} ch  pieces {pieces:>4}  at {record['occurred_at'] or '-':<19} "
            f"flags {','.join(record['flags']) or '-':<16} | {head}")


# Hand-made samples for the kinds the dataset has no fixture for.
SAMPLES = WORK / "samples"
SAMPLES.mkdir(parents=True, exist_ok=True)
(SAMPLES / "meeting-notes.md").write_text(
    "# Weekly sync, 2026-09-02\n\n## Attendees\n\nJustin, Fang\n\n## Decisions\n\n- Kaggle first, desktop later.\n"
    "- The extractor never reads a manifest.\n\n## Actions\n\n- Justin: unpack scripts by Friday.\n", encoding="utf-8")
(SAMPLES / "email.txt").write_text(
    "From: Xing Fang <fang@example.edu>\nTo: Justin Hoffman <justin@example.edu>\nDate: Wed, 3 Sep 2026 09:12:00 -0500\n"
    "Subject: Re: notebook\n\nJustin,\n\nThe chapter notebook looks good. Send the repository link when it is public.\n\nXing\n",
    encoding="utf-8")
(SAMPLES / "transcript.jsonl").write_text(
    json.dumps({"type": "user", "sessionId": "abc123", "timestamp": "2026-09-01T14:02:11Z",
                "message": {"role": "user", "content": "Summarize the Oz book 2 ending."}}) + "\n"
    + json.dumps({"type": "assistant", "sessionId": "abc123", "timestamp": "2026-09-01T14:02:20Z",
                  "message": {"role": "assistant", "content": [{"type": "text", "text": "Tip is revealed to be Ozma."}]}}) + "\n"
    + json.dumps({"type": "queue-operation", "sessionId": "abc123", "timestamp": "2026-09-01T14:02:21Z"}) + "\n",
    encoding="utf-8")

TRIAL = [RAW / "oz" / "01_55.txt", RAW / "oz" / "02_54.txt", RAW / "greek" / "27_library00apolgoog.txt",
         RAW / "greek" / "30_heroidesamores00ovid.txt", RAW / "graphrag-bench" / "Novel-30752.txt",
         RAW / "longmemeval" / "001cefa7_2.json", RAW / "longmemeval" / "sharegpt_yywfIrx_0.json",
         PAPERS / "edge2024-graphrag.pdf", SAMPLES / "meeting-notes.md", SAMPLES / "email.txt",
         SAMPLES / "transcript.jsonl"]
for path in TRIAL:
    print(describe(to_text(path)))
